In [46]:

import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

print("Semua library berhasil dipanggil!")

Semua library berhasil dipanggil!


In [47]:
df = sns.load_dataset("titanic")

print(df.head())
print("Jumlah baris dan kolom:", df.shape)

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
Jumlah baris dan kolom: (891, 15)


In [48]:
fitur = ["age", "fare", "sex", "embarked", "class"]

df = df[fitur + ["survived"]].copy()

X = df[fitur]
y = df["survived"]

print("Fitur yang digunakan:")
print(X.head())

print("Target:")
print(y.head())

Fitur yang digunakan:
    age     fare     sex embarked  class
0  22.0   7.2500    male        S  Third
1  38.0  71.2833  female        C  First
2  26.0   7.9250  female        S  Third
3  35.0  53.1000  female        S  First
4  35.0   8.0500    male        S  Third
Target:
0    0
1    1
2    1
3    1
4    0
Name: survived, dtype: int64


In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

Jumlah data latih: 712
Jumlah data uji: 179


In [50]:
num_cols = ["age", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["class"]

print("Numerik:", num_cols)
print("Kategorikal:", cat_cols)
print("Ordinal:", ord_cols)

Numerik: ['age', 'fare']
Kategorikal: ['sex', 'embarked']
Ordinal: ['class']


In [51]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

ord_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=[["Third", "Second", "First"]]
    ))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
    ("ord", ord_pipe, ord_cols)
])

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

print("Pipeline berhasil dibuat!")

Pipeline berhasil dibuat!


In [52]:
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Akurasi:", round(accuracy, 4))
print("F1-score:", round(f1, 4))

Akurasi: 0.7709
F1-score: 0.6917


In [55]:
def buat_pipeline(num_cols, cat_cols, ord_cols):
    transformers = []

    if num_cols:
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        transformers.append(("num", num_pipe, num_cols))

    if cat_cols:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])
        transformers.append(("cat", cat_pipe, cat_cols))

    if ord_cols:
        ord_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(
                categories=[["Third", "Second", "First"]]
            ))
        ])
        transformers.append(("ord", ord_pipe, ord_cols))

    prep = ColumnTransformer(transformers)

    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(max_iter=1000))
    ])

print("Fungsi berhasil dibuat!")

Fungsi berhasil dibuat!


In [56]:

# MODEL A: menggunakan semua 5 fitur
fitur_semua = ["age", "fare", "sex", "embarked", "class"]

model_semua = buat_pipeline(
    ["age", "fare"],
    ["sex", "embarked"],
    ["class"]
)

model_semua.fit(X_train[fitur_semua], y_train)
pred_semua = model_semua.predict(X_test[fitur_semua])


# MODEL B: menggunakan 3 fitur terpilih
fitur_terpilih = ["age", "fare", "sex"]

model_terpilih = buat_pipeline(
    ["age", "fare"],
    ["sex"],
    []
)

model_terpilih.fit(X_train[fitur_terpilih], y_train)
pred_terpilih = model_terpilih.predict(X_test[fitur_terpilih])


# MEMBANDINGKAN HASIL KEDUA MODEL
hasil = pd.DataFrame({
    "Model": ["Semua fitur", "Fitur terpilih"],
    "Jumlah fitur": [5, 3],
    "Akurasi": [
        accuracy_score(y_test, pred_semua),
        accuracy_score(y_test, pred_terpilih)
    ],
    "F1-score": [
        f1_score(y_test, pred_semua),
        f1_score(y_test, pred_terpilih)
    ]
})

print(hasil.round(4).to_string(index=False))

         Model  Jumlah fitur  Akurasi  F1-score
   Semua fitur             5   0.7709    0.6917
Fitur terpilih             3   0.7765    0.6970


In [62]:
print("=== PIPELINE SEMUA FITUR ===")
pipeline_semua_fitur = buat_pipeline(
    num_cols, cat_cols, ord_cols
)
print(pipeline_semua_fitur)

print("\n=== PIPELINE FITUR TERPILIH ===")
print(model_terpilih)

=== PIPELINE SEMUA FITUR ===
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                             

In [63]:
print("=== DAFTAR SEMUA FITUR ===")
print(num_cols + cat_cols + ord_cols)

print("\n=== DAFTAR FITUR TERPILIH ===")
print(fitur_terpilih)

=== DAFTAR SEMUA FITUR ===
['age', 'fare', 'sex', 'embarked', 'class']

=== DAFTAR FITUR TERPILIH ===
['age', 'fare', 'sex']
